## Necessary import and defination

In [1]:
import cupy as cp
cp.cuda.Device(0).use()
import os
## change the directory to the wholistic_registration code dir, so that we can import the modules in utils
os.chdir('/home/cyf/wbi/Virginia/code/wbi_0123/wholistic_registration/src/wholistic_registration')
from utils import IO
from utils import calFlowCrossResolution, calFlow3d_Wei_v1
from utils import registration
from utils import option
from utils import preprocess as prep
from utils import mask
from utils import  visualization
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/cyf/wbi/Virginia/code/CoarseFlow").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from training.inference import (
    CoarseFlowInferenceConfig,
    CoarseFlowPredictor,
)


F260517_mov_path = "/home/cyf/wbi/Virginia/raw_data/f260517/260517_exp_00001_TZCYX.ome.tiff"
F260517_ref_path = "/home/cyf/wbi/Virginia/raw_data/f260517/260517_anat_00003_TZCYX.ome.tiff"

import numpy as np
import pandas as pd


def summarize_phase_z_offset(
    phase_new,
    z_init,
    fixed_z_planes=None,
    z_window=1.5,
    ref_shape=None,
    ref_volume_order="zyx",
    frame_idx=None,
    valid_xy_only=True,
):
    """
    Summarize z displacement of phase_new relative to initial z per moving slice.

    phase_new:
        shape (X, Y, K, 3), xyz reference coordinates.

    z_init:
        shape (K,), initial reference z for each moving slice.

    fixed_z_planes:
        shape (K,) or (Nplanes,). If length K, compute per-slice coverage
        relative to its corresponding fixed plane.

    z_window:
        slab half-width used for projection.

    ref_shape:
        reference volume shape. Used to compute out-of-bound ratio.
        If ref_volume_order='zyx': ref_shape=(Z,Y,X)
        If ref_volume_order='xyz': ref_shape=(X,Y,Z)

    Returns
    -------
    df:
        pandas DataFrame, one row per moving slice k.
    """

    if hasattr(phase_new, "get"):
        phase = phase_new.get()
    else:
        phase = np.asarray(phase_new)

    z_init = np.asarray(z_init, dtype=np.float32)

    X, Y, K, C = phase.shape
    assert C == 3
    assert len(z_init) == K

    x_ref = phase[..., 0]
    y_ref = phase[..., 1]
    z_ref = phase[..., 2]

    if ref_shape is not None:
        if ref_volume_order == "zyx":
            Zref, Yref, Xref = ref_shape
        elif ref_volume_order == "xyz":
            Xref, Yref, Zref = ref_shape
        else:
            raise ValueError("ref_volume_order should be 'zyx' or 'xyz'.")

        in_xy = (
            np.isfinite(x_ref)
            & np.isfinite(y_ref)
            & (x_ref >= 0) & (x_ref <= Xref - 1)
            & (y_ref >= 0) & (y_ref <= Yref - 1)
        )

        in_z = (
            np.isfinite(z_ref)
            & (z_ref >= 0) & (z_ref <= Zref - 1)
        )

        in_ref = in_xy & in_z
    else:
        in_xy = np.isfinite(x_ref) & np.isfinite(y_ref)
        in_z = np.isfinite(z_ref)
        in_ref = in_xy & in_z

    rows = []

    fixed_z_planes_arr = None
    if fixed_z_planes is not None:
        fixed_z_planes_arr = np.asarray(fixed_z_planes, dtype=np.float32)

    for k in range(K):
        if valid_xy_only:
            valid = in_xy[:, :, k] & np.isfinite(z_ref[:, :, k])
        else:
            valid = in_ref[:, :, k]

        z_k = z_ref[:, :, k]
        dz_k = z_k - float(z_init[k])

        dz_valid = dz_k[valid]

        if dz_valid.size == 0:
            row = {
                "frame": frame_idx,
                "k": k,
                "z_init": float(z_init[k]),
                "n_valid": 0,
                "valid_ratio": 0.0,
                "dz_mean": np.nan,
                "dz_median": np.nan,
                "dz_std": np.nan,
                "dz_p01": np.nan,
                "dz_p05": np.nan,
                "dz_p25": np.nan,
                "dz_p75": np.nan,
                "dz_p95": np.nan,
                "dz_p99": np.nan,
                "dz_min": np.nan,
                "dz_max": np.nan,
                "z_abs_p95_from_init": np.nan,
                "out_of_ref_ratio": np.nan,
            }
        else:
            out_of_ref_ratio = 1.0 - float(np.mean(in_ref[:, :, k]))

            row = {
                "frame": frame_idx,
                "k": k,
                "z_init": float(z_init[k]),
                "n_valid": int(dz_valid.size),
                "valid_ratio": float(dz_valid.size / (X * Y)),
                "dz_mean": float(np.mean(dz_valid)),
                "dz_median": float(np.median(dz_valid)),
                "dz_std": float(np.std(dz_valid)),
                "dz_p01": float(np.percentile(dz_valid, 1)),
                "dz_p05": float(np.percentile(dz_valid, 5)),
                "dz_p25": float(np.percentile(dz_valid, 25)),
                "dz_p75": float(np.percentile(dz_valid, 75)),
                "dz_p95": float(np.percentile(dz_valid, 95)),
                "dz_p99": float(np.percentile(dz_valid, 99)),
                "dz_min": float(np.min(dz_valid)),
                "dz_max": float(np.max(dz_valid)),
                "z_abs_p95_from_init": float(np.percentile(np.abs(dz_valid), 95)),
                "out_of_ref_ratio": out_of_ref_ratio,
            }

        if fixed_z_planes_arr is not None:
            if len(fixed_z_planes_arr) == K:
                z0 = float(fixed_z_planes_arr[k])
                z_dist = np.abs(z_ref[:, :, k] - z0)
                slab_valid = valid & (z_dist <= z_window)

                row["fixed_z"] = z0
                row["coverage_to_own_fixed_plane"] = float(np.mean(slab_valid))
                row["z_dist_to_fixed_median"] = float(np.median(z_dist[valid])) if np.any(valid) else np.nan
                row["z_dist_to_fixed_p95"] = float(np.percentile(z_dist[valid], 95)) if np.any(valid) else np.nan

        rows.append(row)

    return pd.DataFrame(rows)


def estimate_projection_z_from_phase(
    phase_new,
    z_init=None,
    ref_shape=None,
    ref_volume_order="zyx",
    method="trimmed_mean",
    trim_percentiles=(5, 95),
    fallback_to_z_init=True,
    frame_idx=None,
):
    """
    Estimate one target reference z plane for each moving slice k
    from phase_new[..., 2].

    Parameters
    ----------
    phase_new:
        shape (X, Y, K, 3), xyz reference coordinates.

    z_init:
        shape (K,), initial reference z for each moving slice.
        Used for reporting and fallback.

    ref_shape:
        Reference volume shape.
        If ref_volume_order == "zyx": ref_shape = (Z, Y, X)
        If ref_volume_order == "xyz": ref_shape = (X, Y, Z)

    method:
        "mean", "median", or "trimmed_mean".

        I recommend "trimmed_mean" or "median" instead of plain mean,
        because a few wrong z coordinates can shift the average.

    trim_percentiles:
        Used only when method == "trimmed_mean".

    fallback_to_z_init:
        If one slice has no valid coordinates, use z_init[k].

    Returns
    -------
    target_z_pred:
        shape (K,), estimated reference z plane for each moving slice.

    df:
        per-slice diagnostic table.
    """

    import numpy as np
    import pandas as pd

    if hasattr(phase_new, "get"):
        phase = phase_new.get()
    else:
        phase = np.asarray(phase_new)

    if phase.ndim != 4 or phase.shape[-1] != 3:
        raise ValueError(f"phase_new should have shape (X,Y,K,3), got {phase.shape}")

    X, Y, K, _ = phase.shape

    z_ref = phase[..., 2]
    x_ref = phase[..., 0]
    y_ref = phase[..., 1]

    if z_init is not None:
        z_init_arr = np.asarray(z_init, dtype=np.float32)
        if len(z_init_arr) != K:
            raise ValueError(f"z_init length {len(z_init_arr)} does not match K={K}")
    else:
        z_init_arr = np.full(K, np.nan, dtype=np.float32)

    if ref_shape is not None:
        if ref_volume_order == "zyx":
            Zref, Yref, Xref = ref_shape
        elif ref_volume_order == "xyz":
            Xref, Yref, Zref = ref_shape
        else:
            raise ValueError("ref_volume_order should be 'zyx' or 'xyz'.")

        valid = (
            np.isfinite(x_ref)
            & np.isfinite(y_ref)
            & np.isfinite(z_ref)
            & (x_ref >= 0) & (x_ref <= Xref - 1)
            & (y_ref >= 0) & (y_ref <= Yref - 1)
            & (z_ref >= 0) & (z_ref <= Zref - 1)
        )
    else:
        valid = np.isfinite(z_ref)

    target_z_pred = np.zeros(K, dtype=np.float32)
    rows = []

    for k in range(K):
        z_k = z_ref[:, :, k]
        valid_k = valid[:, :, k]

        z_valid = z_k[valid_k]

        if z_valid.size == 0:
            if fallback_to_z_init and z_init is not None:
                z_est = float(z_init_arr[k])
            else:
                z_est = np.nan
        else:
            if method == "mean":
                z_est = float(np.mean(z_valid))

            elif method == "median":
                z_est = float(np.median(z_valid))

            elif method == "trimmed_mean":
                p_low, p_high = trim_percentiles
                lo = np.percentile(z_valid, p_low)
                hi = np.percentile(z_valid, p_high)
                z_trim = z_valid[(z_valid >= lo) & (z_valid <= hi)]

                if z_trim.size == 0:
                    z_est = float(np.median(z_valid))
                else:
                    z_est = float(np.mean(z_trim))
            else:
                raise ValueError("method should be 'mean', 'median', or 'trimmed_mean'.")

        target_z_pred[k] = z_est

        if z_valid.size > 0:
            dz_valid = z_valid - z_init_arr[k]
            rows.append({
                "frame": frame_idx,
                "k": k,
                "z_init": float(z_init_arr[k]),
                "target_z_pred": float(z_est),
                "target_minus_z_init": float(z_est - z_init_arr[k]),
                "z_mean": float(np.mean(z_valid)),
                "z_median": float(np.median(z_valid)),
                "z_p05": float(np.percentile(z_valid, 5)),
                "z_p95": float(np.percentile(z_valid, 95)),
                "dz_mean": float(np.mean(dz_valid)),
                "dz_median": float(np.median(dz_valid)),
                "dz_p05": float(np.percentile(dz_valid, 5)),
                "dz_p95": float(np.percentile(dz_valid, 95)),
                "valid_ratio": float(z_valid.size / (X * Y)),
            })
        else:
            rows.append({
                "frame": frame_idx,
                "k": k,
                "z_init": float(z_init_arr[k]),
                "target_z_pred": float(z_est),
                "target_minus_z_init": float(z_est - z_init_arr[k]) if np.isfinite(z_est) else np.nan,
                "z_mean": np.nan,
                "z_median": np.nan,
                "z_p05": np.nan,
                "z_p95": np.nan,
                "dz_mean": np.nan,
                "dz_median": np.nan,
                "dz_p05": np.nan,
                "dz_p95": np.nan,
                "valid_ratio": 0.0,
            })

    df = pd.DataFrame(rows)
    return target_z_pred, df





zRatio = 10, frame_rate = 4.07066 s/frame

Test to do the registration
---

#### helper functions

In [2]:
from importlib import reload
from typing import Any

from numpy import ndarray
import numpy as np

reload(calFlowCrossResolution)
reload(prep)

F260517_mov, F260517_mov_desc = IO.readTiff(F260517_mov_path)
F260517_ref, F260517_ref_desc = IO.readTiff(F260517_ref_path)
print("Finish reading the dataset")

from collections import deque
import numpy as np
import pandas as pd


def robust_average_target_z(target_z_list, method="median", trim_percentiles=(10, 90)):
    """
    Combine target_z estimates from multiple warmup frames.

    target_z_list:
        list of arrays, each shape (K,)

    return:
        fixed_target_z, shape (K,)
    """
    arr = np.stack([np.asarray(z, dtype=np.float32) for z in target_z_list], axis=0)
    # arr: (Nwarmup, K)

    if method == "median":
        fixed_z = np.nanmedian(arr, axis=0)

    elif method == "mean":
        fixed_z = np.nanmean(arr, axis=0)

    elif method == "trimmed_mean":
        lo_p, hi_p = trim_percentiles
        fixed = []
        for k in range(arr.shape[1]):
            vals = arr[:, k]
            vals = vals[np.isfinite(vals)]
            if vals.size == 0:
                fixed.append(np.nan)
                continue
            lo = np.percentile(vals, lo_p)
            hi = np.percentile(vals, hi_p)
            vals_trim = vals[(vals >= lo) & (vals <= hi)]
            if vals_trim.size == 0:
                fixed.append(float(np.median(vals)))
            else:
                fixed.append(float(np.mean(vals_trim)))
        fixed_z = np.asarray(fixed, dtype=np.float32)

    else:
        raise ValueError("method should be 'median', 'mean', or 'trimmed_mean'.")

    return fixed_z.astype(np.float32)


def make_phase_pool_from_records(records):
    """
    Build a pooled phase field from nearby frames.

    Each record should contain:
        record["phase_new"]: shape (X, Y, K, 3)

    Output:
        phase_pool: shape (X, Y, K * Nrecords, 3)

    This lets the projection function use nearby frame surfaces together.
    """
    phases = [np.asarray(r["phase_new"], dtype=np.float32) for r in records]
    return np.concatenate(phases, axis=2)


def get_nearby_records(records, center_idx, radius):
    """
    Select records whose frame index is within [center_idx-radius, center_idx+radius].
    """
    out = []
    for r in records:
        if abs(int(r["frame"]) - int(center_idx)) <= radius:
            out.append(r)
    return out


def update_reference_intensity_mapping_from_target_stack(
    F260517_ref_mem,
    target_stack_zyx,
    z_idx,
    option,
    thresFactor,
    maskRange,
    smoothPenalty_raw,
    percentiles,
):
    """
    Learn raw reference -> target intensity mapping using only z_idx planes,
    then apply the learned mapping to the full reference volume.

    Parameters
    ----------
    F260517_ref_mem:
        Raw reference membrane volume, shape (Zref, Y, X).

    target_stack_zyx:
        Target stack used for learning intensity mapping, shape (K, Y, X).

        Initial update:
            target_stack_zyx = raw moving frame 0, F260517_mov_mem[0]

        Later updates:
            target_stack_zyx = previous registered/mapped membrane image,
            i.e. mem_mapped_prev.transpose(2, 1, 0)

    z_idx:
        Reference z planes corresponding to moving slices, shape (K,).

    Returns
    -------
    F260517_ref_mem_adj:
        Adjusted full reference volume, shape (X, Y, Zref).

    src_q, tgt_q, used_percentiles:
        Quantile mapping information.
    """

    target_stack_zyx = np.asarray(target_stack_zyx, dtype=np.float32)

    if target_stack_zyx.ndim != 3:
        raise ValueError(
            f"target_stack_zyx should be (K,Y,X), got {target_stack_zyx.shape}"
        )

    if target_stack_zyx.shape[0] != len(z_idx):
        raise ValueError(
            f"target_stack_zyx K={target_stack_zyx.shape[0]} does not match "
            f"z_idx length={len(z_idx)}"
        )

    # ------------------------------------------------------------
    # 1. Learn mapping only on matched planes.
    # ------------------------------------------------------------
    ref_source_zyx = F260517_ref_mem[z_idx].astype(np.float32, copy=False)

    if ref_source_zyx.shape != target_stack_zyx.shape:
        raise ValueError(
            f"ref_source_zyx shape {ref_source_zyx.shape} does not match "
            f"target_stack_zyx shape {target_stack_zyx.shape}"
        )

    src_q, tgt_q, used_percentiles = prep.learn_quantile_mapping(
        source=ref_source_zyx,
        target=target_stack_zyx,
        percentiles=percentiles,
    )

    # ------------------------------------------------------------
    # 2. Apply learned mapping to the full reference volume.
    # ------------------------------------------------------------
    F260517_ref_mem_adj = prep.apply_quantile_mapping(
        F260517_ref_mem,
        src_q,
        tgt_q,
    ).transpose(2, 1, 0).astype(np.float32, copy=False)  # (X, Y, Zref)

    # ------------------------------------------------------------
    # 3. Update reference mask and smooth penalty.
    # ------------------------------------------------------------
    option["mask_ref"] = mask.getMask(F260517_ref_mem_adj, thresFactor)
    option["mask_ref"] = mask.bwareafilt3_wei(option["mask_ref"], maskRange)

    Pnltfactor = prep.getSmPnltNormFctr(F260517_ref_mem_adj, option)
    option["smoothPenalty"] = Pnltfactor * smoothPenalty_raw

    return F260517_ref_mem_adj, src_q, tgt_q, used_percentiles


def project_and_save_frame(
    frame_idx,
    records_for_projection,
    fixed_target_z,
    F260517_ref_mem_adj,
    F260517_ref_sparseCell,
    out_dir,
):
    """
    Use nearby-frame phase fields as a pooled surface and project reference signals
    to fixed_target_z.
    """
    phase_pool = make_phase_pool_from_records(records_for_projection)

    mov_nanmax_proj = calFlowCrossResolution.project_coords_to_fixed_planes_gpu(
        coords_ref_xyk_xyz=phase_pool,
        ref_volume=F260517_ref_mem_adj,
        target_z_planes=fixed_target_z,
        values_xyk=None,
        ref_volume_order="xyz",
        z_window=projection_z_window,
        downsample_xy=projection_downsample_xy,
        fill_value=projection_fill_value,
        return_numpy=True,
        output_order="zyx",
        xy_splat_mode="subpixel_footprint",
        xy_extra_radius=projection_xy_extra_radius,
    )

    sparseCell_mapped_nanmax_proj = calFlowCrossResolution.project_coords_to_fixed_planes_gpu(
        coords_ref_xyk_xyz=phase_pool,
        ref_volume=F260517_ref_sparseCell,
        target_z_planes=fixed_target_z,
        values_xyk=None,
        ref_volume_order="zyx",
        z_window=projection_z_window,
        downsample_xy=projection_downsample_xy,
        fill_value=projection_fill_value,
        return_numpy=True,
        output_order="zyx",
        xy_splat_mode="subpixel_footprint",
        xy_extra_radius=projection_xy_extra_radius,
    )

    IO.write_multichannel_volume_as_ome_tiff(
        volume=[
            mov_nanmax_proj,
            sparseCell_mapped_nanmax_proj,
        ],
        out_dir=out_dir,
        frame_idx=frame_idx,
        label="F260517",
    )




Finish reading the dataset


#### base config

In [3]:
########### Split into 2 stacks
F260517_ref_mem = F260517_ref[90:310, 1, :, :]
F260517_ref_sparseCell = F260517_ref[90:310, 0, :, :]

F260517_mov_mem: ndarray[tuple[int, ...], Any] = F260517_mov[:, :, 1, :, :]
F260517_mov_sparseCell: ndarray[tuple[int, ...], Any] = F260517_mov[:, :, 0, :, :]

print("F260517_ref_mem:", F260517_ref_mem.shape)          # (Zref, Y, X)
print("F260517_mov_mem:", F260517_mov_mem.shape)          # (T, K, Y, X)
print("Finish splitting the dataset")


########### Registration options
option['r'] = 5
option['layer'] = 3
option['iter'] = 10
option['movRange'] = 5.
option['tol'] = 1e-6
option['zRatio_HR'] = 1
option["wrong_region_enable"] = False

thresFactor = 5.
maskRange = [5., 4000.]
smoothPenalty_raw = 0.01


########### Find the corresponding z slice
z_init = calFlowCrossResolution.FindInitZ_stack_global_fixed_spacing(
    F260517_mov_mem[0].transpose(2, 1, 0),     # (X, Y, K)
    F260517_ref_mem.transpose(2, 1, 0),        # (X, Y, Zref)
    delta_ref_idx=10,
    use_gradient=False
)
z_init = np.asarray(z_init, dtype=np.float32)
z_idx = np.rint(z_init).astype(np.int32)
z_idx = np.clip(z_idx, 0, F260517_ref_mem.shape[0] - 1)

K = z_init.shape[0]

x = np.arange(F260517_mov_mem[0].shape[2], dtype=np.float32)
y = np.arange(F260517_mov_mem[0].shape[1], dtype=np.float32)
k = np.arange(K, dtype=np.int32)

X_grid, Y_grid, K_grid = np.meshgrid(
    x,
    y,
    k,
    indexing="ij",
)

coords_xyz = np.empty(
    (F260517_mov_mem[0].shape[2], F260517_mov_mem[0].shape[1], K, 3),
    dtype=np.float32
)

coords_xyz[..., 0] = X_grid
coords_xyz[..., 1] = Y_grid
coords_xyz[..., 2] = z_init[K_grid]

option["phase"] = coords_xyz

print("z_init:", z_init)
print("z_idx:", z_idx)
print("phase shape:", option["phase"].shape)

########### Quantile mapping helper
percentiles = [
    0.1, 0.5, 1, 2, 5,
    10, 25, 50, 75, 90,
    95, 99, 99.5, 99.8
]

########### Reference update setting
update_every = 40
update_mode = "mean_last5"

all_z_stats = []
########### Registration
T = F260517_mov_mem.shape[0]

# =========================================================
# Projection settings
# =========================================================
warmup_n = 5                     
target_z_combine_method = "median"  # "median", "mean", or "trimmed_mean"

projection_z_window = 5.0
projection_downsample_xy = 2
projection_fill_value = -200.0
projection_xy_extra_radius = 1

out_dir = "/home/cyf/wbi/Virginia/registrated_data/f260517/f260517_0529_refined/"
os.makedirs(out_dir, exist_ok=True)
records = []

target_z_pred_list = []
all_z_plane_stats = []
all_z_offset_stats = []

fixed_target_z = None
has_flushed_warmup = False


F260517_ref_mem: (220, 1500, 630)
F260517_mov_mem: (200, 20, 1500, 630)
Finish splitting the dataset
z_init: [ 20.  30.  40.  50.  60.  70.  80.  90. 100. 110. 120. 130. 140. 150.
 160. 170. 180. 190. 200. 210.]
z_idx: [ 20  30  40  50  60  70  80  90 100 110 120 130 140 150 160 170 180 190
 200 210]
phase shape: (630, 1500, 20, 3)


#### run the loop

In [4]:
# =========================================================
# Initial reference intensity mapping
# =========================================================
from collections import deque

F260517_ref_mem_adj, src_q, tgt_q, used_percentiles = (
    update_reference_intensity_mapping_from_target_stack(
        F260517_ref_mem=F260517_ref_mem,
        target_stack_zyx=F260517_mov_mem[0].astype(np.float32),  # (K, Y, X)
        z_idx=z_idx,
        option=option,
        thresFactor=thresFactor,
        maskRange=maskRange,
        smoothPenalty_raw=smoothPenalty_raw,
        percentiles=percentiles,
    )
)

print("[Init] Updated reference intensity mapping using raw moving frame 0")
print("src_q:", src_q)
print("tgt_q:", tgt_q)


# =========================================================
# Settings
# =========================================================
T = F260517_mov_mem.shape[0]

# Reference intensity mapping update
update_every = 40
update_mode = "mapped_recent_mean"     # "mapped_previous" or "mapped_recent_mean"
mapped_update_window = 5
mapped_mem_buffer = deque(maxlen=mapped_update_window)

# Projection plane estimation
# fixed_target_z is estimated from the first warmup_n frames.
warmup_n = 5
target_z_combine_method = "median"     # "median", "mean", or "trimmed_mean"

# Projection settings
# Important:
#   projection uses ONLY the current frame's phase_new.
#   No neighboring-frame phase pooling is used.
projection_z_window = 5.0
projection_downsample_xy = 2
projection_fill_value = -200.0
projection_xy_extra_radius = 1

out_dir = "/home/cyf/wbi/Virginia/registrated_data/f260517/f260517_0529_refined/"
os.makedirs(out_dir, exist_ok=True)

# Warmup records only.
# After fixed_target_z is determined, records will be flushed and cleared.
records = []
target_z_pred_list = []
all_z_plane_stats = []

fixed_target_z = None

# Keep reference-adjusted volumes by mapping version.
# This avoids using a new F260517_ref_mem_adj to project old warmup frames.
mapping_version = 0
ref_mem_adj_by_version = {mapping_version: F260517_ref_mem_adj}


# =========================================================
# Target-z estimation helper
# =========================================================
def estimate_projection_z_from_phase_simple(
    phase_new,
    z_init,
    ref_shape,
    ref_volume_order="zyx",
    method="trimmed_mean",
    trim_percentiles=(5, 95),
    frame_idx=None,
):
    """
    Estimate one target reference z plane for each moving slice k
    from phase_new[..., 2].

    Return:
        target_z_pred: (K,)
        df: per-slice diagnostics
    """
    if hasattr(phase_new, "get"):
        phase = phase_new.get()
    else:
        phase = np.asarray(phase_new)

    phase = np.asarray(phase, dtype=np.float32)
    z_init = np.asarray(z_init, dtype=np.float32)

    if phase.ndim != 4 or phase.shape[-1] != 3:
        raise ValueError(f"phase_new should be (X,Y,K,3), got {phase.shape}")

    X, Y, K, _ = phase.shape

    if len(z_init) != K:
        raise ValueError(f"z_init length {len(z_init)} does not match K={K}")

    x_ref = phase[..., 0]
    y_ref = phase[..., 1]
    z_ref = phase[..., 2]

    if ref_volume_order == "zyx":
        Zref, Yref, Xref = ref_shape
    elif ref_volume_order == "xyz":
        Xref, Yref, Zref = ref_shape
    else:
        raise ValueError("ref_volume_order should be 'zyx' or 'xyz'.")

    valid = (
        np.isfinite(x_ref)
        & np.isfinite(y_ref)
        & np.isfinite(z_ref)
        & (x_ref >= 0) & (x_ref <= Xref - 1)
        & (y_ref >= 0) & (y_ref <= Yref - 1)
        & (z_ref >= 0) & (z_ref <= Zref - 1)
    )

    target_z_pred = np.zeros(K, dtype=np.float32)
    rows = []

    for k_idx in range(K):
        z_k = z_ref[:, :, k_idx]
        valid_k = valid[:, :, k_idx]
        z_valid = z_k[valid_k]

        if z_valid.size == 0:
            z_est = float(z_init[k_idx])
        else:
            if method == "mean":
                z_est = float(np.mean(z_valid))

            elif method == "median":
                z_est = float(np.median(z_valid))

            elif method == "trimmed_mean":
                p_low, p_high = trim_percentiles
                lo = np.percentile(z_valid, p_low)
                hi = np.percentile(z_valid, p_high)
                z_trim = z_valid[(z_valid >= lo) & (z_valid <= hi)]

                if z_trim.size == 0:
                    z_est = float(np.median(z_valid))
                else:
                    z_est = float(np.mean(z_trim))
            else:
                raise ValueError("method should be 'mean', 'median', or 'trimmed_mean'.")

        target_z_pred[k_idx] = z_est

        if z_valid.size > 0:
            dz_valid = z_valid - float(z_init[k_idx])
            rows.append({
                "frame": frame_idx,
                "k": k_idx,
                "z_init": float(z_init[k_idx]),
                "target_z_pred": float(z_est),
                "target_minus_z_init": float(z_est - z_init[k_idx]),
                "z_median": float(np.median(z_valid)),
                "z_mean": float(np.mean(z_valid)),
                "z_p05": float(np.percentile(z_valid, 5)),
                "z_p95": float(np.percentile(z_valid, 95)),
                "dz_median": float(np.median(dz_valid)),
                "dz_p05": float(np.percentile(dz_valid, 5)),
                "dz_p95": float(np.percentile(dz_valid, 95)),
                "valid_ratio": float(z_valid.size / (X * Y)),
            })
        else:
            rows.append({
                "frame": frame_idx,
                "k": k_idx,
                "z_init": float(z_init[k_idx]),
                "target_z_pred": float(z_est),
                "target_minus_z_init": float(z_est - z_init[k_idx]),
                "z_median": np.nan,
                "z_mean": np.nan,
                "z_p05": np.nan,
                "z_p95": np.nan,
                "dz_median": np.nan,
                "dz_p05": np.nan,
                "dz_p95": np.nan,
                "valid_ratio": 0.0,
            })

    return target_z_pred, pd.DataFrame(rows)


# =========================================================
# Fixed-z averaging helper
# =========================================================
def robust_average_target_z(target_z_list, method="median", trim_percentiles=(10, 90)):
    """
    Combine target_z estimates from multiple warmup frames.

    target_z_list:
        list of arrays, each shape (K,)

    return:
        fixed_target_z, shape (K,)
    """
    arr = np.stack([np.asarray(z, dtype=np.float32) for z in target_z_list], axis=0)

    if method == "median":
        fixed_z = np.nanmedian(arr, axis=0)

    elif method == "mean":
        fixed_z = np.nanmean(arr, axis=0)

    elif method == "trimmed_mean":
        lo_p, hi_p = trim_percentiles
        fixed = []

        for k_idx in range(arr.shape[1]):
            vals = arr[:, k_idx]
            vals = vals[np.isfinite(vals)]

            if vals.size == 0:
                fixed.append(np.nan)
                continue

            lo = np.percentile(vals, lo_p)
            hi = np.percentile(vals, hi_p)
            vals_trim = vals[(vals >= lo) & (vals <= hi)]

            if vals_trim.size == 0:
                fixed.append(float(np.median(vals)))
            else:
                fixed.append(float(np.mean(vals_trim)))

        fixed_z = np.asarray(fixed, dtype=np.float32)

    else:
        raise ValueError("method should be 'median', 'mean', or 'trimmed_mean'.")

    return fixed_z.astype(np.float32)


# =========================================================
# Single-frame projection helper
# =========================================================
def project_and_save_single_frame(
    frame_idx,
    phase_new,
    fixed_target_z,
    ref_mem_adj_for_projection,
    F260517_ref_sparseCell,
    out_dir,
):
    """
    Project one frame using ONLY this frame's phase_new.

    Important:
        fixed_target_z can be estimated from warmup frames,
        but frame i's projection must use frame i's own phase_new only.
    """
    phase_new = np.asarray(phase_new, dtype=np.float32)

    mem_proj = calFlowCrossResolution.project_coords_to_fixed_planes_gpu(
        coords_ref_xyk_xyz=phase_new,
        ref_volume=ref_mem_adj_for_projection,
        target_z_planes=fixed_target_z,
        values_xyk=None,
        ref_volume_order="xyz",
        z_window=projection_z_window,
        downsample_xy=projection_downsample_xy,
        fill_value=projection_fill_value,
        return_numpy=True,
        output_order="zyx",
        xy_splat_mode="subpixel_footprint",
        xy_extra_radius=projection_xy_extra_radius,
    )

    sparse_proj = calFlowCrossResolution.project_coords_to_fixed_planes_gpu(
        coords_ref_xyk_xyz=phase_new,
        ref_volume=F260517_ref_sparseCell,
        target_z_planes=fixed_target_z,
        values_xyk=None,
        ref_volume_order="zyx",
        z_window=projection_z_window,
        downsample_xy=projection_downsample_xy,
        fill_value=projection_fill_value,
        return_numpy=True,
        output_order="zyx",
        xy_splat_mode="subpixel_footprint",
        xy_extra_radius=projection_xy_extra_radius,
    )

    IO.write_multichannel_volume_as_ome_tiff(
        volume=[
            mem_proj,
            sparse_proj,
        ],
        out_dir=out_dir,
        frame_idx=frame_idx,
        label="F260517",
    )

    # Optional memory cleanup if you defined cleanup_gpu_memory elsewhere.
    if "cleanup_gpu_memory" in globals():
        cleanup_gpu_memory()


# =========================================================
# Main registration loop
# =========================================================
for i in range(T):

    if i > 75:
        option["wrong_region_enable"] = True

    # ------------------------------------------------------------
    # 1. Current moving frame
    # ------------------------------------------------------------
    F260517_mov_mem_i = F260517_mov_mem[i].transpose(2, 1, 0).astype(
        np.float32,
        copy=False,
    )  # (X, Y, K)

    # ------------------------------------------------------------
    # 2. Moving mask
    # ------------------------------------------------------------
    option["mask_mov"] = mask.getMask(F260517_mov_mem_i, thresFactor)
    option["mask_mov"] = mask.bwareafilt3_wei(option["mask_mov"], maskRange)

    # ------------------------------------------------------------
    # 3. Registration
    # ------------------------------------------------------------
    phase_new, motion_current, mem_mapped = calFlowCrossResolution.getMotion_v2(
        F260517_mov_mem_i,
        F260517_ref_mem_adj,
        option,
        verbose=False,
    )

    # Make sure these are CPU numpy arrays.
    if hasattr(phase_new, "get"):
        phase_new = phase_new.get()
    if hasattr(motion_current, "get"):
        motion_current = motion_current.get()
    if hasattr(mem_mapped, "get"):
        mem_mapped = mem_mapped.get()

    phase_new = phase_new.astype(np.float32, copy=False)
    motion_current = motion_current.astype(np.float32, copy=False)
    mem_mapped = mem_mapped.astype(np.float32, copy=False)

    mem_err = float(np.mean(np.abs(F260517_mov_mem_i - mem_mapped)))

    print("=" * 80)
    print(f"Frame {i}")
    print("Mem channel error:", mem_err)
    print("mapping_version:", mapping_version)

    # ------------------------------------------------------------
    # 4. Cache registered/mapped membrane for intensity mapping update
    # ------------------------------------------------------------
    # mem_mapped: (X, Y, K)
    # target_stack_for_update should be (K, Y, X)
    last_mem_mapped_zyx = mem_mapped.transpose(2, 1, 0).astype(
        np.float32,
        copy=False,
    )

    mapped_mem_buffer.append(last_mem_mapped_zyx)

    # ------------------------------------------------------------
    # 5. Temporal initialization for next frame
    # ------------------------------------------------------------
    option["motion"] = (0.7 * motion_current).astype(np.float32, copy=False)

    # ------------------------------------------------------------
    # 6. Estimate target z for this frame
    # ------------------------------------------------------------
    target_z_pred_i, df_z_plane_i = estimate_projection_z_from_phase_simple(
        phase_new=phase_new,
        z_init=z_init,
        ref_shape=F260517_ref_mem.shape,  # raw ref: (Z, Y, X)
        ref_volume_order="zyx",
        method="trimmed_mean",
        trim_percentiles=(5, 95),
        frame_idx=i,
    )

    target_z_pred_list.append(target_z_pred_i)
    all_z_plane_stats.append(df_z_plane_i)

    # ------------------------------------------------------------
    # 7. Build current record
    # ------------------------------------------------------------
    current_record = {
        "frame": i,
        "phase_new": phase_new,
        "target_z_pred": target_z_pred_i.astype(np.float32, copy=False),
        "mem_err": mem_err,
        "mapping_version": mapping_version,
    }

    # ------------------------------------------------------------
    # 8. Determine fixed target z after warmup
    # ------------------------------------------------------------
    just_determined_fixed_z = False

    if fixed_target_z is None:
        records.append(current_record)

        if len(target_z_pred_list) >= warmup_n:
            fixed_target_z = robust_average_target_z(
                target_z_pred_list[:warmup_n],
                method=target_z_combine_method,
                trim_percentiles=(10, 90),
            )

            bad = ~np.isfinite(fixed_target_z)
            fixed_target_z[bad] = z_init[bad]

            just_determined_fixed_z = True

            print("=" * 80)
            print(f"[Projection] fixed_target_z determined using first {warmup_n} frames")
            print("z_init:", z_init)
            print("fixed_target_z:", fixed_target_z)
            print("fixed_target_z - z_init:", fixed_target_z - z_init)
            print("=" * 80)

    # ------------------------------------------------------------
    # 9. Projection saving
    #
    # Rule:
    #   - fixed_target_z is estimated from warmup frames.
    #   - frame i projection uses ONLY frame i's phase_new.
    # ------------------------------------------------------------
    if fixed_target_z is None:
        print(f"[Projection] warmup collecting: {len(target_z_pred_list)}/{warmup_n}")

    else:
        if just_determined_fixed_z:
            # Flush warmup frames.
            for rec in records:
                frame_to_save = int(rec["frame"])
                rec_version = int(rec["mapping_version"])

                ref_mem_adj_for_projection = ref_mem_adj_by_version[rec_version]

                project_and_save_single_frame(
                    frame_idx=frame_to_save,
                    phase_new=rec["phase_new"],
                    fixed_target_z=fixed_target_z,
                    ref_mem_adj_for_projection=ref_mem_adj_for_projection,
                    F260517_ref_sparseCell=F260517_ref_sparseCell,
                    out_dir=out_dir,
                )

                print(
                    f"[Projection] saved warmup frame {frame_to_save} | "
                    f"mapping_version={rec_version}"
                )

            # Projection no longer needs old frames after warmup.
            records.clear()

        else:
            # Save current frame immediately.
            rec_version = int(current_record["mapping_version"])
            ref_mem_adj_for_projection = ref_mem_adj_by_version[rec_version]

            project_and_save_single_frame(
                frame_idx=i,
                phase_new=current_record["phase_new"],
                fixed_target_z=fixed_target_z,
                ref_mem_adj_for_projection=ref_mem_adj_for_projection,
                F260517_ref_sparseCell=F260517_ref_sparseCell,
                out_dir=out_dir,
            )

            print(
                f"[Projection] saved frame {i} | "
                f"mapping_version={rec_version}"
            )

    # ------------------------------------------------------------
    # 10. Reference intensity mapping update
    #
    # Important:
    # after finishing frame i=39, update mapping for frame 40 onward.
    # Source: raw F260517_ref_mem[z_idx]
    # Target: previous/recent registered mem_mapped stack, shape (K,Y,X)
    # ------------------------------------------------------------
    should_update_ref = ((i + 1) % update_every == 0) and ((i + 1) < T)

    if should_update_ref:

        if update_mode == "mapped_previous":
            target_stack_for_update = last_mem_mapped_zyx
            update_info = f"registered mem_mapped from frame {i}"

        elif update_mode == "mapped_recent_mean":
            if len(mapped_mem_buffer) == 0:
                raise RuntimeError(
                    "mapped_mem_buffer is empty; cannot update reference mapping."
                )

            target_stack_for_update = np.mean(
                np.stack(list(mapped_mem_buffer), axis=0),
                axis=0,
            ).astype(np.float32, copy=False)

            update_info = (
                f"mean registered mem_mapped from recent "
                f"{len(mapped_mem_buffer)} frames ending at frame {i}"
            )

        else:
            raise ValueError(
                f"Unknown update_mode: {update_mode}. "
                "Use 'mapped_previous' or 'mapped_recent_mean'."
            )

        F260517_ref_mem_adj, src_q, tgt_q, used_percentiles = (
            update_reference_intensity_mapping_from_target_stack(
                F260517_ref_mem=F260517_ref_mem,
                target_stack_zyx=target_stack_for_update,
                z_idx=z_idx,
                option=option,
                thresFactor=thresFactor,
                maskRange=maskRange,
                smoothPenalty_raw=smoothPenalty_raw,
                percentiles=percentiles,
            )
        )

        mapping_version += 1
        ref_mem_adj_by_version[mapping_version] = F260517_ref_mem_adj

        print("=" * 80)
        print(f"[After frame {i}] Updated reference intensity mapping using {update_info}")
        print("new mapping_version:", mapping_version)
        print("target_stack_for_update shape:", target_stack_for_update.shape)
        print("ref_source shape:", F260517_ref_mem[z_idx].shape)
        print("src_q:", src_q)
        print("tgt_q:", tgt_q)
        print("=" * 80)


# =========================================================
# Fallback: if T < warmup_n, determine fixed_target_z from all available frames
# and save cached frames.
# =========================================================
if fixed_target_z is None and len(records) > 0:
    fixed_target_z = robust_average_target_z(
        target_z_pred_list,
        method=target_z_combine_method,
        trim_percentiles=(10, 90),
    )

    bad = ~np.isfinite(fixed_target_z)
    fixed_target_z[bad] = z_init[bad]

    print("=" * 80)
    print(f"[Projection] fixed_target_z determined using all available {len(records)} frames")
    print("z_init:", z_init)
    print("fixed_target_z:", fixed_target_z)
    print("fixed_target_z - z_init:", fixed_target_z - z_init)
    print("=" * 80)

    for rec in records:
        frame_to_save = int(rec["frame"])
        rec_version = int(rec["mapping_version"])
        ref_mem_adj_for_projection = ref_mem_adj_by_version[rec_version]

        project_and_save_single_frame(
            frame_idx=frame_to_save,
            phase_new=rec["phase_new"],
            fixed_target_z=fixed_target_z,
            ref_mem_adj_for_projection=ref_mem_adj_for_projection,
            F260517_ref_sparseCell=F260517_ref_sparseCell,
            out_dir=out_dir,
        )

        print(
            f"[Projection] saved fallback frame {frame_to_save} | "
            f"mapping_version={rec_version}"
        )

    records.clear()


# =========================================================
# Save diagnostics
# =========================================================
if len(all_z_plane_stats) > 0:
    z_plane_stats_all = pd.concat(all_z_plane_stats, ignore_index=True)
    z_plane_stats_all.to_csv(
        os.path.join(out_dir, "z_plane_pred_stats.csv"),
        index=False,
    )

if fixed_target_z is not None:
    np.save(
        os.path.join(out_dir, "fixed_target_z.npy"),
        fixed_target_z.astype(np.float32),
    )

print("Done.")

[Init] Updated reference intensity mapping using raw moving frame 0
src_q: [ -188.     -156.     -140.     -120.      -87.      -53.       26.
   314.     1897.     4546.     5872.     8854.    10194.    11971.002]
tgt_q: [-195. -170. -157. -141. -114.  -86.  -23.  158. 1072. 2578. 3352. 5064.
 5848. 6917.]
Frame 0
Mem channel error: 172.50753784179688
mapping_version: 0
[Projection] warmup collecting: 1/5
Frame 1
Mem channel error: 172.708740234375
mapping_version: 0
[Projection] warmup collecting: 2/5
Frame 2
Mem channel error: 173.09817504882812
mapping_version: 0
[Projection] warmup collecting: 3/5
Frame 3
Mem channel error: 172.82025146484375
mapping_version: 0
[Projection] warmup collecting: 4/5
Frame 4
Mem channel error: 172.60906982421875
mapping_version: 0
[Projection] fixed_target_z determined using first 5 frames
z_init: [ 20.  30.  40.  50.  60.  70.  80.  90. 100. 110. 120. 130. 140. 150.
 160. 170. 180. 190. 200. 210.]
fixed_target_z: [ 20.729996  30.820724  40.96902   49

## Test to do the registration with Model output

In [2]:
def phase_kyx_xyz_to_xyk_xyz(phase_kyx_xyz):
    """
    Convert CoarseFlow phase to getMotion_v2 phase.

    Input:
        phase_kyx_xyz: (K, Y, X, 3), xyz

    Output:
        phase_xyk_xyz: (X, Y, K, 3), xyz
    """
    phase_kyx_xyz = np.asarray(phase_kyx_xyz, dtype=np.float32)

    if phase_kyx_xyz.ndim != 4 or phase_kyx_xyz.shape[-1] != 3:
        raise ValueError(
            f"phase_kyx_xyz should be (K,Y,X,3), got {phase_kyx_xyz.shape}"
        )

    return phase_kyx_xyz.transpose(2, 1, 0, 3).astype(np.float32, copy=False)

def predict_coarse_phase_for_frame(
    coarse_predictor,
    mov_mem_zyx,
    ref_mem_adj_xyz,
    z_init,
    ref_spacing=(1.0, 1.0, 1.0),
    normalize=True,
):
    """
    Predict coarse phase for one frame.

    Args:
        mov_mem_zyx:
            moving memory channel, shape (K,Y,X)

        ref_mem_adj_xyz:
            adjusted reference memory channel, shape (X,Y,Zref)

        z_init:
            initial z for each moving slice, shape (K,)

    Returns:
        coarse_phase_xyk_xyz:
            phase for getMotion_v2, shape (X,Y,K,3), xyz
    """
    mov_mem_zyx = np.asarray(mov_mem_zyx, dtype=np.float32)
    ref_mem_adj_xyz = np.asarray(ref_mem_adj_xyz, dtype=np.float32)
    z_init = np.asarray(z_init, dtype=np.float32).reshape(-1)

    # CoarseFlow expects ref as (Z,Y,X)
    ref_mem_adj_zyx = ref_mem_adj_xyz.transpose(2, 1, 0).astype(
        np.float32,
        copy=False,
    )

    coarse_phase_kyx_xyz = coarse_predictor.predict_full(
        mov=mov_mem_zyx,
        ref=ref_mem_adj_zyx,
        z_init=z_init,
        ref_spacing=ref_spacing,

        mov_order="zyx",
        ref_order="zyx",

        # 建议 True：模型输入做 robust normalization；
        # refine 仍然使用原来的 ref_mem_adj_xyz，不受影响。
        normalize=normalize,

        phase_order="xyz",
        return_dict=False,
    )

    coarse_phase_xyk_xyz = phase_kyx_xyz_to_xyk_xyz(coarse_phase_kyx_xyz)

    return coarse_phase_xyk_xyz

def prepare_ref_mov_for_coarse_and_refine(
    F_ref_mem_zyx,
    target_mov_zyx,
    z_idx,
    prep,
    percentiles,
    norm_p_low=0.001,
    norm_p_high=0.999,
    eps=1e-6,
):
    """
    Prepare intensity-aligned inputs for CoarseFlow and getMotion_v2.

    Args:
        F_ref_mem_zyx:
            raw reference memory stack, shape (Zref,Y,X)

        target_mov_zyx:
            current moving sparse stack, shape (K,Y,X)

        z_idx:
            initial reference slice indices, shape (K,)

    Returns:
        dict with:
            ref_adj_zyx:
                quantile-mapped reference, shape (Zref,Y,X)

            ref_adj_xyz:
                quantile-mapped reference for getMotion_v2, shape (X,Y,Zref)

            mov_norm_zyx:
                moving stack normalized by mov-derived stats, shape (K,Y,X)

            ref_adj_norm_zyx:
                adjusted reference normalized by same mov-derived stats, shape (Zref,Y,X)

            src_q, tgt_q, used_percentiles:
                quantile mapping parameters
    """
    F_ref_mem_zyx = np.asarray(F_ref_mem_zyx, dtype=np.float32)
    target_mov_zyx = np.asarray(target_mov_zyx, dtype=np.float32)
    z_idx = np.asarray(z_idx, dtype=np.int32)

    if F_ref_mem_zyx.ndim != 3:
        raise ValueError(f"F_ref_mem_zyx should be (Z,Y,X), got {F_ref_mem_zyx.shape}")

    if target_mov_zyx.ndim != 3:
        raise ValueError(f"target_mov_zyx should be (K,Y,X), got {target_mov_zyx.shape}")

    if target_mov_zyx.shape[0] != len(z_idx):
        raise ValueError(
            f"target_mov_zyx K={target_mov_zyx.shape[0]} "
            f"does not match z_idx length={len(z_idx)}"
        )

    z_idx = np.clip(z_idx, 0, F_ref_mem_zyx.shape[0] - 1)

    # ------------------------------------------------------------
    # 1. Learn ref -> mov quantile mapping from corresponding slices
    # ------------------------------------------------------------
    ref_source_zyx = F_ref_mem_zyx[z_idx]  # (K,Y,X)

    src_q, tgt_q, used_percentiles = prep.learn_quantile_mapping(
        source=ref_source_zyx,
        target=target_mov_zyx,
        percentiles=percentiles,
    )

    ref_adj_zyx = prep.apply_quantile_mapping(
        F_ref_mem_zyx,
        src_q,
        tgt_q,
    ).astype(np.float32, copy=False)

    # ------------------------------------------------------------
    # 2. Learn robust normalization from moving image
    #    and apply SAME normalization to mov and adjusted ref.
    # ------------------------------------------------------------
    vals = target_mov_zyx[np.isfinite(target_mov_zyx)]
    vals = vals[vals != 0] if np.sum(vals != 0) > 100 else vals

    lo = np.percentile(vals, norm_p_low * 100.0)
    hi = np.percentile(vals, norm_p_high * 100.0)

    if not np.isfinite(lo):
        lo = float(np.nanmin(target_mov_zyx))
    if (not np.isfinite(hi)) or hi <= lo:
        hi = lo + 1.0

    mov_norm_zyx = np.clip(
        (target_mov_zyx - lo) / (hi - lo + eps),
        0.0,
        1.0,
    ).astype(np.float32, copy=False)

    ref_adj_norm_zyx = np.clip(
        (ref_adj_zyx - lo) / (hi - lo + eps),
        0.0,
        1.0,
    ).astype(np.float32, copy=False)

    return {
        "ref_adj_zyx": ref_adj_zyx,
        "ref_adj_xyz": ref_adj_zyx.transpose(2, 1, 0).astype(np.float32, copy=False),

        "mov_norm_zyx": mov_norm_zyx,
        "ref_adj_norm_zyx": ref_adj_norm_zyx,

        "src_q": src_q,
        "tgt_q": tgt_q,
        "used_percentiles": used_percentiles,
        "norm_lo": float(lo),
        "norm_hi": float(hi),
        "z_idx": z_idx,
    }

def normalize_mov_and_ref_with_mov_stats(
    mov_zyx,
    ref_adj_zyx,
    p_low=0.001,
    p_high=0.999,
    mask_nonzero=True,
    sample_voxels=2_000_000,
    eps=1e-6,
    seed=12345,
):
    """
    Learn robust normalization statistics from current moving stack,
    then apply the same normalization to both mov and adjusted reference.

    mov_zyx:
        current moving stack, shape (K,Y,X)

    ref_adj_zyx:
        quantile-mapped reference stack, shape (Z,Y,X)

    return:
        mov_norm_zyx
        ref_adj_norm_zyx
        norm_info
    """
    mov_zyx = np.asarray(mov_zyx, dtype=np.float32)
    ref_adj_zyx = np.asarray(ref_adj_zyx, dtype=np.float32)

    vals = mov_zyx[np.isfinite(mov_zyx)]

    if mask_nonzero:
        vals_nz = vals[vals != 0]
        if vals_nz.size > 100:
            vals = vals_nz

    if vals.size == 0:
        lo, hi = 0.0, 1.0
    else:
        if vals.size > sample_voxels:
            rng = np.random.default_rng(seed)
            idx = rng.choice(vals.size, size=sample_voxels, replace=False)
            vals = vals[idx]

        lo = float(np.percentile(vals, p_low * 100.0))
        hi = float(np.percentile(vals, p_high * 100.0))

        if not np.isfinite(lo):
            lo = 0.0
        if (not np.isfinite(hi)) or hi <= lo:
            hi = lo + 1.0

    mov_norm_zyx = np.clip(
        (mov_zyx - lo) / (hi - lo + eps),
        0.0,
        1.0,
    ).astype(np.float32, copy=False)

    ref_adj_norm_zyx = np.clip(
        (ref_adj_zyx - lo) / (hi - lo + eps),
        0.0,
        1.0,
    ).astype(np.float32, copy=False)

    norm_info = {
        "lo": lo,
        "hi": hi,
        "p_low": p_low,
        "p_high": p_high,
        "mask_nonzero": mask_nonzero,
    }

    return mov_norm_zyx, ref_adj_norm_zyx, norm_info

def predict_coarse_phase_for_frame_preprocessed(
    coarse_predictor,
    mov_norm_zyx,
    ref_adj_norm_zyx,
    z_init,
    ref_spacing=(1.0, 1.0, 1.0, 1.0, 1.0, 10.0),
):
    """
    Predict coarse phase using already normalized inputs.

    mov_norm_zyx:
        (K,Y,X), normalized current moving stack

    ref_adj_norm_zyx:
        (Z,Y,X), adjusted reference normalized by the same mov-derived stats

    return:
        coarse_phase_xyk_xyz:
            (X,Y,K,3), xyz, directly usable as option["phase"]
    """
    coarse_phase_kyx_xyz = coarse_predictor.predict_full(
        mov=mov_norm_zyx,
        ref=ref_adj_norm_zyx,
        z_init=z_init,
        ref_spacing=ref_spacing,

        mov_order="zyx",
        ref_order="zyx",

        # Important: already normalized.
        normalize=False,

        phase_order="xyz",
        return_dict=False,
    )

    coarse_phase_xyk_xyz = phase_kyx_xyz_to_xyk_xyz(coarse_phase_kyx_xyz)

    return coarse_phase_xyk_xyz


In [ ]:
from importlib import reload
from typing import Any

from numpy import ndarray
import numpy as np

reload(calFlowCrossResolution)
reload(prep)

pth_path = "/home/cyf/wbi/Virginia/code/CoarseFlow/scripys/checkpoints/coarse_matching_v0527/best.pth"

cfg = CoarseFlowInferenceConfig(
    pth_path=pth_path,
    model_config=None,
    model_module="models.SparseGMFlow3D_v2",
    model_class="CoarseMatchingNetV6",
    device="cuda:0",      
    strict_load=True,
    use_amp=False,
)

coarse_predictor = CoarseFlowPredictor(cfg)

print("Loaded CoarseFlow predictor.")
print("load_msg:", coarse_predictor.load_msg)
print("control_stride:", coarse_predictor.model_config.get("control_stride", None))


F260517_mov, F260517_mov_desc = IO.readTiff(F260517_mov_path)
F260517_ref, F260517_ref_desc = IO.readTiff(F260517_ref_path)
print("Finish reading the dataset")


########### Split into 2 stacks
F260517_ref_mem = F260517_ref[90:310, 1, :, :]
F260517_ref_sparseCell = F260517_ref[90:310, 0, :, :]

F260517_mov_mem: ndarray[tuple[int, ...], Any] = F260517_mov[:, :, 1, :, :]
F260517_mov_sparseCell: ndarray[tuple[int, ...], Any] = F260517_mov[:, :, 0, :, :]

print("F260517_ref_mem:", F260517_ref_mem.shape)          # (Zref, Y, X)
print("F260517_mov_mem:", F260517_mov_mem.shape)          # (T, K, Y, X)
print("Finish splitting the dataset")


########### Registration options
option["r"] = 5
option["layer"] = 3
option["iter"] = 10
option["movRange"] = 5.0
option["tol"] = 1e-6

# Required by getMotion_v2
option["zRatio"] = 1.0
option["zRatio_HR"] = 1.0

option["wrong_region_enable"] = False

thresFactor = 5.0
maskRange = [5.0, 4000.0]
smoothPenalty_raw = 0.01



########### CoarseFlow setting
use_coarse_init = True
coarse_every = 1
# V6 spacing embedding expects 6 values:
# [sx_ref, sy_ref, sz_ref, sx_mov, sy_mov, sz_mov]
delta_ref_idx = 10.0
coarse_spacing6 = (1.0, 1.0, 1.0, 1.0, 1.0, float(delta_ref_idx))
# Whether to remove previous-frame motion before getMotion_v2.
reset_previous_motion_when_using_coarse = True
# Debug logs
coarse_norm_log = []
coarse_phase_log = []


########### Find the corresponding z slice
z_init = calFlowCrossResolution.FindInitZ_stack_global_fixed_spacing(
    F260517_mov_mem[0].transpose(2, 1, 0),     # (X, Y, K)
    F260517_ref_mem.transpose(2, 1, 0),        # (X, Y, Zref)
    delta_ref_idx=10,
    use_gradient=False
)

z_init = np.asarray(z_init, dtype=np.float32)
z_idx = np.rint(z_init).astype(np.int32)
z_idx = np.clip(z_idx, 0, F260517_ref_mem.shape[0] - 1)

K = z_init.shape[0]

x = np.arange(F260517_mov_mem[0].shape[2], dtype=np.float32)
y = np.arange(F260517_mov_mem[0].shape[1], dtype=np.float32)
k = np.arange(K, dtype=np.int32)

X_grid, Y_grid, K_grid = np.meshgrid(
    x,
    y,
    k,
    indexing="ij",
)

coords_xyz = np.empty(
    (F260517_mov_mem[0].shape[2], F260517_mov_mem[0].shape[1], K, 3),
    dtype=np.float32
)

coords_xyz[..., 0] = X_grid
coords_xyz[..., 1] = Y_grid
coords_xyz[..., 2] = z_init[K_grid]

option["phase"] = coords_xyz

print("z_init:", z_init)
print("z_idx:", z_idx)
print("phase shape:", option["phase"].shape)


########### Quantile mapping helper
percentiles = [
    0.1, 0.5, 1, 2, 5,
    10, 25, 50, 75, 90,
    95, 99, 99.5, 99.8
]


def update_reference_intensity_mapping(
    F260517_ref_mem,
    target_mov_zyx,
    z_idx,
    option,
    thresFactor,
    maskRange,
    smoothPenalty_raw,
    percentiles,
):
    """
    Learn ref -> target_mov intensity mapping and update:
        1. F260517_ref_mem_adj
        2. option['mask_ref']
        3. option['smoothPenalty']

    F260517_ref_mem: (Zref, Y, X)
    target_mov_zyx:  (K, Y, X)
    z_idx:           (K,)
    """

    target_mov_zyx = np.asarray(target_mov_zyx, dtype=np.float32)

    if target_mov_zyx.ndim != 3:
        raise ValueError(f"target_mov_zyx should be (K,Y,X), got {target_mov_zyx.shape}")

    if target_mov_zyx.shape[0] != len(z_idx):
        raise ValueError(
            f"target_mov_zyx K={target_mov_zyx.shape[0]} does not match z_idx length={len(z_idx)}"
        )

    ref_source_zyx = F260517_ref_mem[z_idx]  # (K, Y, X)

    src_q, tgt_q, used_percentiles = prep.learn_quantile_mapping(
        source=ref_source_zyx,
        target=target_mov_zyx,
        percentiles=percentiles,
    )

    F260517_ref_mem_adj = prep.apply_quantile_mapping(
        F260517_ref_mem,
        src_q,
        tgt_q
    ).transpose(2, 1, 0).astype(np.float32, copy=False)  # (X, Y, Zref)

    option['mask_ref'] = mask.getMask(F260517_ref_mem_adj, thresFactor)
    option['mask_ref'] = mask.bwareafilt3_wei(option['mask_ref'], maskRange)

    Pnltfactor = prep.getSmPnltNormFctr(F260517_ref_mem_adj, option)
    option['smoothPenalty'] = Pnltfactor * smoothPenalty_raw

    return F260517_ref_mem_adj, src_q, tgt_q, used_percentiles


########### Initial reference adjustment using frame 0
F260517_ref_mem_adj, src_q, tgt_q, used_percentiles = update_reference_intensity_mapping(
    F260517_ref_mem=F260517_ref_mem,
    target_mov_zyx=F260517_mov_mem[0],
    z_idx=z_idx,
    option=option,
    thresFactor=thresFactor,
    maskRange=maskRange,
    smoothPenalty_raw=smoothPenalty_raw,
    percentiles=percentiles,
)

print("[Init] Updated reference intensity mapping using frame 0")
print("src_q:", src_q)
print("tgt_q:", tgt_q)


########### Generate H for sparseCell reference
H_sparseCell = calFlowCrossResolution.generate_continuous_H_gpu(
    F260517_ref_sparseCell.transpose(2, 1, 0),
    zRatio=1
)


########### Reference update setting
update_every = 40
update_mode = "single_frame"


########### Registration
########### Registration
T = F260517_mov_mem.shape[0]

for i in range(T):

    if i > 75:
        option["wrong_region_enable"] = True

    # ------------------------------------------------------------
    # 0. Current moving frame
    # ------------------------------------------------------------
    mov_mem_zyx = F260517_mov_mem[i].astype(np.float32, copy=False)          # (K,Y,X)
    mov_sparse_zyx = F260517_mov_sparseCell[i].astype(np.float32, copy=False)

    F260517_mov_mem_i = mov_mem_zyx.transpose(2, 1, 0)                       # (X,Y,K)
    F260517_mov_sparseCell_i = mov_sparse_zyx.transpose(2, 1, 0)             # (X,Y,K)

    # ------------------------------------------------------------
    # 1. CoarseFlow initialization
    # ------------------------------------------------------------
    run_coarse_this_frame = (
        use_coarse_init
        and (coarse_every is not None)
        and (i % coarse_every == 0)
    )

    if run_coarse_this_frame:
        # F260517_ref_mem_adj is for getMotion_v2: (X,Y,Zref)
        # CoarseFlow needs reference as (Zref,Y,X)
        ref_adj_zyx = F260517_ref_mem_adj.transpose(2, 1, 0).astype(
            np.float32,
            copy=False,
        )

        # Learn normalization from current moving stack,
        # and apply the SAME normalization to adjusted reference.
        mov_norm_zyx, ref_adj_norm_zyx, norm_info = normalize_mov_and_ref_with_mov_stats(
            mov_zyx=mov_mem_zyx,
            ref_adj_zyx=ref_adj_zyx,
            p_low=0.01,
            p_high=0.99,
            mask_nonzero=True,
        )

        # Predict coarse phase using preprocessed inputs.
        # Output should be (X,Y,K,3), xyz, directly usable by getMotion_v2.
        coarse_phase_xyk_xyz = predict_coarse_phase_for_frame_preprocessed(
            coarse_predictor=coarse_predictor,
            mov_norm_zyx=mov_norm_zyx,
            ref_adj_norm_zyx=ref_adj_norm_zyx,
            z_init=z_init,
            ref_spacing=coarse_spacing6,
        )
        if i >0:
            option["phase"] = 0.3 * coarse_phase_xyk_xyz + 0.7 * (coords_xyz + 0.7 * motion_current )
        else:
            option["phase"] = coarse_phase_xyk_xyz
        # Avoid double initialization:
        # coarse phase already gives an initial coordinate field.
        if reset_previous_motion_when_using_coarse and ("motion" in option):
            option.pop("motion")

        # Optional debug: inspect coarse displacement magnitude
        # z0 = z_init.reshape(1, 1, -1)                                      # for (X,Y,K)
        # dx = coarse_phase_xyk_xyz[..., 0] - coords_xyz[..., 0]
        # dy = coarse_phase_xyk_xyz[..., 1] - coords_xyz[..., 1]
        # dz = coarse_phase_xyk_xyz[..., 2] - coords_xyz[..., 2]

        # coarse_phase_log.append(
        #     {
        #         "frame": i,
        #         "dx_mean": float(np.mean(dx)),
        #         "dy_mean": float(np.mean(dy)),
        #         "dz_mean": float(np.mean(dz)),
        #         "dx_abs_mean": float(np.mean(np.abs(dx))),
        #         "dy_abs_mean": float(np.mean(np.abs(dy))),
        #         "dz_abs_mean": float(np.mean(np.abs(dz))),
        #         "dx_abs_p95": float(np.percentile(np.abs(dx), 95)),
        #         "dy_abs_p95": float(np.percentile(np.abs(dy), 95)),
        #         "dz_abs_p95": float(np.percentile(np.abs(dz), 95)),
        #     }
        # )

        # coarse_norm_log.append(
        #     {
        #         "frame": i,
        #         "lo": float(norm_info["lo"]),
        #         "hi": float(norm_info["hi"]),
        #     }
        # )

        # if i % 10 == 0:
        #     print(
        #         f"[Coarse] frame={i} | "
        #         f"norm_lo={norm_info['lo']:.4g}, norm_hi={norm_info['hi']:.4g} | "
        #         f"|dx|={np.mean(np.abs(dx)):.3f}, "
        #         f"|dy|={np.mean(np.abs(dy)):.3f}, "
        #         f"|dz|={np.mean(np.abs(dz)):.3f}"
        #     )

    else:
        # Baseline initialization: initial z + identity xy
        option["phase"] = coords_xyz.copy()

    # ------------------------------------------------------------
    # 2. Moving mask from raw moving image
    # ------------------------------------------------------------
    option["mask_mov"] = mask.getMask(F260517_mov_mem_i, thresFactor)
    option["mask_mov"] = mask.bwareafilt3_wei(option["mask_mov"], maskRange)

    # ------------------------------------------------------------
    # 3. Refine using getMotion_v2
    #    Important:
    #    - mov uses raw current moving image: (X,Y,K)
    #    - ref uses quantile-mapped adjusted reference: (X,Y,Zref)
    #    - option["phase"] is either coarse phase or initial phase
    # ------------------------------------------------------------
    phase_new, motion_current, mem_mapped = calFlowCrossResolution.getMotion_v2(
        F260517_mov_mem_i,
        F260517_ref_mem_adj,
        option,
        verbose=False,
    )

    # ------------------------------------------------------------
    # 4. Apply final refined phase to sparse-cell channel
    # ------------------------------------------------------------
    sparseCell_mapped = calFlowCrossResolution.apply_H_to_matrix_gpu(
        phase_new,
        H_sparseCell,
    ).get()

    # ------------------------------------------------------------
    # 5. Print errors
    # ------------------------------------------------------------
    mem_abs_err = float(np.mean(np.abs(F260517_mov_mem_i - mem_mapped)))
    sparse_abs_err = float(np.mean(np.abs(F260517_mov_sparseCell_i - sparseCell_mapped)))

    print(f"Frame {i}")
    print("Mem channel error: ", mem_abs_err)
    print("Sparse cell channel error: ", sparse_abs_err)

    # ------------------------------------------------------------
    # 6. Temporal motion update
    # ------------------------------------------------------------
    #
    # if not run_coarse_this_frame:
    #     option["motion"] = 0.7 * motion_current

    # if coarse_every > 1:
    # option["motion"] = 0.7 * motion_current

    # ------------------------------------------------------------
    # 7. Save outputs
    # ------------------------------------------------------------
    IO.write_multichannel_volume_as_ome_tiff(
        volume=[
            F260517_mov_mem_i[::2, ::2, :].transpose(2, 1, 0),
            mem_mapped[::2, ::2, :].transpose(2, 1, 0),
            F260517_mov_sparseCell_i[::2, ::2, :].transpose(2, 1, 0),
            sparseCell_mapped[::2, ::2, :].transpose(2, 1, 0),
        ],
        out_dir="/home/cyf/wbi/Virginia/registrated_data/f260517/f260517_0527/",
        frame_idx=i,
        label="F260517",
    )

    # ------------------------------------------------------------
    # 8. Reference intensity mapping update
    # ------------------------------------------------------------
    should_update_ref = ((i + 1) % update_every == 0) and ((i + 1) < T)

    if should_update_ref:
        if update_mode == "single_frame":
            target_mov_for_update = F260517_mov_mem[i].astype(np.float32)
            update_info = f"single frame {i}"

        elif update_mode == "mean_last5":
            start_idx = max(0, i - 4)
            end_idx = i + 1

            target_mov_for_update = np.mean(
                F260517_mov_mem[start_idx:end_idx].astype(np.float32),
                axis=0,
            )

            update_info = f"mean raw mov frames {start_idx}-{end_idx - 1}"

        else:
            raise ValueError(f"Unknown update_mode: {update_mode}")

        F260517_ref_mem_adj, src_q, tgt_q, used_percentiles = update_reference_intensity_mapping(
            F260517_ref_mem=F260517_ref_mem,
            target_mov_zyx=target_mov_for_update,
            z_idx=z_idx,
            option=option,
            thresFactor=thresFactor,
            maskRange=maskRange,
            smoothPenalty_raw=smoothPenalty_raw,
            percentiles=percentiles,
        )

        print("=" * 80)
        print(f"[After frame {i}] Updated reference intensity mapping using {update_info}")
        print("src_q:", src_q)
        print("tgt_q:", tgt_q)
        print("=" * 80)